In [ ]:
from pathlib import Path
from PIL import Image, ImageOps
import csv

# 실제 dataset 폴더 경로로 수정
INPUT_DIR = Path.cwd() / "dataset"

# 결과는 원본 폴더 옆의 dataset_640에 저장
OUTPUT_DIR = INPUT_DIR.with_name("dataset_640")

SIZE = 640
EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}

if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"입력 폴더가 없습니다: {INPUT_DIR}")

if OUTPUT_DIR.exists():
    raise FileExistsError(
        f"결과 폴더가 이미 있습니다. OUTPUT_DIR을 다른 이름으로 바꾸세요: {OUTPUT_DIR}"
    )

files = sorted(
    p for p in INPUT_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in EXTENSIONS
)

if not files:
    raise ValueError("이미지를 찾을 수 없습니다.")

# 저장하기 전에 폴더 구조와 파일명 충돌 확인
output_names = set()

for path in files:
    relative = path.relative_to(INPUT_DIR)

    if len(relative.parts) < 2:
        raise ValueError(f"이미지를 클래스명 폴더 안에 넣으세요: {relative}")

    if path.with_suffix(".txt").exists():
        raise ValueError(
            f"정답박스 라벨이 있습니다. 박스 좌표도 함께 변환해야 합니다: {path}"
        )

    name = str(relative.with_suffix(".png")).casefold()

    if name in output_names:
        raise ValueError(f"PNG로 바꾸면 파일명이 겹칩니다: {relative}")

    output_names.add(name)

OUTPUT_DIR.mkdir(parents=True)
records = []

for index, path in enumerate(files, start=1):
    relative = path.relative_to(INPUT_DIR)

    with Image.open(path) as raw:
        image = ImageOps.exif_transpose(raw).convert("RGBA")

    original_width, original_height = image.size

    # 투명한 부분은 흰색으로 처리
    background = Image.new("RGBA", image.size, "white")
    background.alpha_composite(image)
    image = background.convert("RGB")

    # 큰 이미지만 비율 유지 축소, 작은 이미지는 확대하지 않음
    image.thumbnail((SIZE, SIZE), Image.Resampling.LANCZOS)

    width, height = image.size
    left = (SIZE - width) // 2
    top = (SIZE - height) // 2

    # 흰 여백을 넣어 정확히 640×640으로 맞춤
    result = Image.new("RGB", (SIZE, SIZE), "white")
    result.paste(image, (left, top))

    # 클래스별 폴더 구조 유지
    output_path = OUTPUT_DIR / relative.with_suffix(".png")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.save(output_path, format="PNG")

    records.append({
        "class_name": relative.parts[0],
        "source": str(relative),
        "output": str(output_path.relative_to(OUTPUT_DIR)),
        "original_width": original_width,
        "original_height": original_height,
        "resized_width": width,
        "resized_height": height,
        "scale_x": width / original_width,
        "scale_y": height / original_height,
        "padding_left": left,
        "padding_top": top,
    })

    print(f"[{index}/{len(files)}] {relative} → 640×640")

# 나중에 좌표 변환에 사용할 기록 저장
with (OUTPUT_DIR / "resize_manifest.csv").open(
    "w", encoding="utf-8-sig", newline=""
) as file:
    writer = csv.DictWriter(file, fieldnames=records[0].keys())
    writer.writeheader()
    writer.writerows(records)

print(f"\n완료: {len(files)}장")
print(f"저장 위치: {OUTPUT_DIR}")